In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import subprocess
import sys

# Don't pin versions — let Kaggle use what's already installed
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers",
    "datasets",
    "tqdm"
])

import tensorflow as tf
import numpy as np
import sklearn
import transformers
import torch

print(f"TensorFlow:   {tf.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"PyTorch:      {torch.__version__}")

# GPU setup
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"\nGPUs detected: {len(gpus)}")
print("Environment ready ✅")

TensorFlow:   2.19.0
NumPy:        2.4.6
scikit-learn: 1.6.1
transformers: 5.0.0
PyTorch:      2.10.0+cu128

GPUs detected: 2
Environment ready ✅


In [4]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.utils import resample
from collections import Counter

# Load GoEmotions
print("Loading GoEmotions dataset...")
dataset = load_dataset("go_emotions", "simplified")

label_names = [
    'admiration','amusement','anger','annoyance','approval',
    'caring','confusion','curiosity','desire','disappointment',
    'disapproval','disgust','embarrassment','excitement','fear',
    'gratitude','grief','joy','love','nervousness','optimism',
    'pride','realization','relief','remorse','sadness','surprise','neutral'
]

emotion_mapping = {
    'admiration':'Confident','amusement':'Bored','anger':'Frustrated',
    'annoyance':'Frustrated','approval':'Confident','caring':'Bored',
    'confusion':'Confused','curiosity':'Curious','desire':'Bored',
    'disappointment':'Frustrated','disapproval':'Frustrated','disgust':'Frustrated',
    'embarrassment':'Bored','excitement':'Curious','fear':'Frustrated',
    'gratitude':'Confident','grief':'Bored','joy':'Confident','love':'Bored',
    'nervousness':'Frustrated','optimism':'Confident','pride':'Confident',
    'realization':'Confused','relief':'Confident','remorse':'Frustrated',
    'sadness':'Bored','surprise':'Confused','neutral':'Bored'
}

def remap_labels(example):
    if len(example['labels']) == 0:
        example['emotion'] = 'Bored'
    else:
        original = label_names[example['labels'][0]]
        example['emotion'] = emotion_mapping[original]
    return example

train_data = dataset['train'].map(remap_labels)
val_data   = dataset['validation'].map(remap_labels)
test_data  = dataset['test'].map(remap_labels)

# Convert to DataFrame
train_df = pd.DataFrame({'text': [x['text'] for x in train_data],
                          'emotion': [x['emotion'] for x in train_data]})
val_df   = pd.DataFrame({'text': [x['text'] for x in val_data],
                          'emotion': [x['emotion'] for x in val_data]})
test_df  = pd.DataFrame({'text': [x['text'] for x in test_data],
                          'emotion': [x['emotion'] for x in test_data]})

# Balance to 4000 per class
TARGET = 4000
balanced_dfs = []
for emotion in ['Bored','Confident','Confused','Curious','Frustrated']:
    subset = train_df[train_df['emotion'] == emotion]
    resampled = resample(subset,
                         n_samples=TARGET,
                         replace=len(subset) < TARGET,
                         random_state=42)
    balanced_dfs.append(resampled)

train_df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print("GoEmotions loaded and balanced!")
print(f"\nTrain distribution:")
print(train_df['emotion'].value_counts())
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Loading GoEmotions dataset...


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions loaded and balanced!

Train distribution:
emotion
Confused      4000
Bored         4000
Curious       4000
Frustrated    4000
Confident     4000
Name: count, dtype: int64

Train: 20000 | Val: 5426 | Test: 5427


In [5]:
import re
import nltk
import pickle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
keep_words = {'not','no','never',"don't","can't","won't",
              "doesn't","isn't","aren't","very","really"}
stop_words = stop_words - keep_words

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)

print("Cleaning texts...")
train_df['clean_text'] = train_df['text'].apply(clean_text)
val_df['clean_text']   = val_df['text'].apply(clean_text)
test_df['clean_text']  = test_df['text'].apply(clean_text)

# Label encoding
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['emotion'])
val_df['label']   = le.transform(val_df['emotion'])
test_df['label']  = le.transform(test_df['emotion'])

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Label mapping:")
for i, emotion in enumerate(le.classes_):
    print(f"  {i} → {emotion}")

# Tokenization — owner's exact hyperparameters
MAX_VOCAB_SIZE = 30000
MAX_SEQ_LEN    = 80

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['clean_text'])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_text']),
                         maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
X_val   = pad_sequences(tokenizer.texts_to_sequences(val_df['clean_text']),
                         maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
X_test  = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_text']),
                         maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

y_train = np.array(train_df['label'])
y_val   = np.array(val_df['label'])
y_test  = np.array(test_df['label'])

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print(f"\nShapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")
print("\nPREPROCESSING COMPLETE ✅")

Cleaning texts...
Label mapping:
  0 → Bored
  1 → Confident
  2 → Confused
  3 → Curious
  4 → Frustrated

Shapes:
  X_train: (20000, 80)
  X_val:   (5426, 80)
  X_test:  (5427, 80)

PREPROCESSING COMPLETE ✅


In [9]:
# Cell 4 - BiLSTM With Focal Loss (Fixed)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional,
                                      LSTM, Dense, Dropout,
                                      SpatialDropout1D)
from tensorflow.keras.optimizers import Adam

def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=5)
        cross_entropy = -y_true_one_hot * tf.math.log(
            tf.clip_by_value(y_pred, 1e-7, 1.0)
        )
        weight = alpha * y_true_one_hot * tf.pow(
            1 - tf.clip_by_value(y_pred, 1e-7, 1.0), gamma
        )
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss_fn

tf.keras.backend.clear_session()

EMBED_DIM   = 128
LSTM_UNITS  = 128
NUM_CLASSES = 5

baseline_model = Sequential([
    # mask_zero=False fixes the cuDNN conflict
    Embedding(input_dim=MAX_VOCAB_SIZE,
              output_dim=EMBED_DIM,
              mask_zero=False),      # ← THIS was the problem
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(LSTM_UNITS,
                       dropout=0.2)), # no mask_zero = cuDNN works fine
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

baseline_model.compile(
    optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)

baseline_model.summary()
print("\nBaseline BiLSTM ready ✅")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Baseline BiLSTM ready ✅


In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import time

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=2,
    factor=0.5,
    verbose=1
)

print("Training BiLSTM baseline on GoEmotions...")
start = time.time()

history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=512,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

elapsed = (time.time() - start) / 60
print(f"\nTraining time: {elapsed:.2f} minutes")
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")

# Save baseline
baseline_model.save('baseline_bilstm.keras')
print("Baseline model saved! ✅")

Training BiLSTM baseline on GoEmotions...
Epoch 1/10


I0000 00:00:1782662298.408415     159 cuda_dnn.cc:529] Loaded cuDNN version 91002


40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.2740 - loss: 0.2504 - val_accuracy: 0.3247 - val_loss: 0.2245 - learning_rate: 0.0010
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.4974 - loss: 0.1844 - val_accuracy: 0.4427 - val_loss: 0.1909 - learning_rate: 0.0010
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.6643 - loss: 0.1218 - val_accuracy: 0.4869 - val_loss: 0.1788 - learning_rate: 0.0010
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.7562 - loss: 0.0868 - val_accuracy: 0.4854 - val_loss: 0.1916 - learning_rate: 0.0010
Epoch 5/10
39/40 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8234 - loss: 0.0626
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.8159 - loss: 0.0648 - val_accuracy: 0.4853 - val_loss: 0.2150 - learning_rate: 0.0010
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.8601 - loss: 0.0491 - val_accuracy: 0.493

In [14]:
# Cell 6 - Final BiLSTM On Custom Data With Proper Evaluation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional, LSTM,
                                      Dense, Dropout, SpatialDropout1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

tf.keras.backend.clear_session()

# Proper train/val split from custom data
X_tr, X_va, y_tr, y_va = train_test_split(
    X_custom, y_custom,
    test_size=0.2,
    random_state=42,
    stratify=y_custom
)

print(f"Custom train: {len(X_tr)} | Custom val: {len(X_va)}")

# Build model
bilstm_final = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=False),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])

bilstm_final.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = bilstm_final.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=60,
    batch_size=8,
    callbacks=[
        EarlyStopping(monitor='val_accuracy',
                      patience=8,
                      restore_best_weights=True,
                      verbose=1)
    ],
    verbose=1
)

best_val = max(history.history['val_accuracy'])
print(f"\nBest val accuracy: {best_val:.4f}")

# Full classification report
y_pred = np.argmax(bilstm_final.predict(X_va), axis=1)
print("\nClassification Report:")
print(classification_report(y_va, y_pred,
      target_names=le.classes_))

# Save
bilstm_final.save('bilstm_final.keras')
print("\nBiLSTM saved! ✅")

Custom train: 200 | Custom val: 50
Epoch 1/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.2000 - loss: 1.6142 - val_accuracy: 0.3400 - val_loss: 1.6038
Epoch 2/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2450 - loss: 1.6009 - val_accuracy: 0.4200 - val_loss: 1.5901
Epoch 3/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4150 - loss: 1.5587 - val_accuracy: 0.5200 - val_loss: 1.5395
Epoch 4/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5500 - loss: 1.3804 - val_accuracy: 0.3400 - val_loss: 1.3851
Epoch 5/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6450 - loss: 1.0136 - val_accuracy: 0.4400 - val_loss: 1.2367
Epoch 6/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7800 - loss: 0.6316 - val_accuracy: 0.4800 - val_loss: 1.2323
Epoch 7/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8550 - loss: 0.3623 - val_accuracy: 0.5600 - val_loss: 1.1580
Epoch 8/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9750 - loss

In [15]:
# Cell 7 - BERT Training
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Same custom data split as BiLSTM — fair comparison
texts  = list(custom_df['text'].values)
labels = list(custom_df['label'].values)

texts_train, texts_val, y_tr, y_va = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"BERT Train: {len(texts_train)} | BERT Val: {len(texts_val)}")

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=80):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = EmotionDataset(texts_train, y_tr, bert_tokenizer)
val_dataset   = EmotionDataset(texts_val,   y_va, bert_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# Load BERT
print("\nLoading BERT...")
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=5,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3,
    ignore_mismatched_sizes=True
)
bert_model = bert_model.to(device)

EPOCHS      = 15
optimizer   = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 5,
    num_training_steps=total_steps
)

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0
    loop = tqdm(loader, desc="  Training", leave=False)
    for batch in loop:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        preds = torch.argmax(outputs.logits, dim=1)
        total_correct  += (preds == labels).sum().item()
        total_loss     += outputs.loss.item()
        total_samples  += labels.size(0)
        loop.set_postfix(loss=f"{outputs.loss.item():.4f}")
    return total_loss/len(loader), total_correct/total_samples

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            preds = torch.argmax(outputs.logits, dim=1)
            total_correct  += (preds == labels).sum().item()
            total_loss     += outputs.loss.item()
            total_samples  += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), total_correct/total_samples, all_preds, all_labels

# Training loop
print("\nStarting BERT fine-tuning...")
print(f"Epochs: {EPOCHS} | Device: {device}")
print("─" * 50)

best_val_acc  = 0
best_preds    = []
best_labels   = []
no_improve    = 0
PATIENCE      = 5

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(
        bert_model, train_loader, optimizer, scheduler, device
    )
    val_loss, val_acc, preds, labels_out = eval_epoch(
        bert_model, val_loader, device
    )
    print(f"  train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f}")
    print(f"  val_loss:   {val_loss:.4f}   | val_acc:   {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_preds   = preds
        best_labels  = labels_out
        torch.save(bert_model.state_dict(), 'bert_model.pt')
        print(f"  ✅ Best saved! Val Acc: {val_acc:.4f}")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break

print("\n" + "─" * 50)
print(f"BERT Training Complete!")
print(f"Best Val Accuracy: {best_val_acc:.4f}")

# Final classification report
print("\nClassification Report (Best Epoch):")
print(classification_report(
    best_labels, best_preds,
    target_names=le.classes_
))

Device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT Train: 200 | BERT Val: 50
Train batches: 13
Val batches:   4

Loading BERT...


...

  train_loss: 1.0001 | train_acc: 0.6250
  val_loss:   1.0474   | val_acc:   0.7000
  ✅ Best saved! Val Acc: 0.7000

Epoch 12/15


  train_loss: 0.9182 | train_acc: 0.7100
  val_loss:   1.0124   | val_acc:   0.6200

Epoch 13/15


  train_loss: 0.9102 | train_acc: 0.6900
  val_loss:   1.0053   | val_acc:   0.6400

Epoch 14/15


  train_loss: 0.8344 | train_acc: 0.7350
  val_loss:   0.9820   | val_acc:   0.6800

Epoch 15/15


  train_loss: 0.8596 | train_acc: 0.7400
  val_loss:   0.9758   | val_acc:   0.6800

──────────────────────────────────────────────────
BERT Training Complete!
Best Val Accuracy: 0.7000

Classification Report (Best Epoch):
              precision    recall  f1-score   support

       Bored       0.60      0.60      0.60        10
   Confident       0.59      1.00      0.74        10
    Confused       0.50      0.10      0.17        10
     Curious       0.77      1.00      0.87        10
  Frustrated       1.00      0.80      0.89        10

    accuracy                           0.70        50
   macro avg       0.69      0.70      0.65        50
weighted avg       0.69      0.70      0.65        50



In [16]:
# Cell 8 - Verify All Saved Files
import os
import torch

# Save bert model config too
from transformers import BertConfig
config = BertConfig.from_pretrained('bert-base-uncased', num_labels=5)
config.save_pretrained('bert_config/')

files = [
    'label_encoder.pkl',
    'tokenizer.pkl',
    'baseline_bilstm.keras',
    'bilstm_final.keras',
    'bert_model.pt'
]

print("Verifying saved files:")
all_good = True
for f in files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024 if exists else 0
    status = '✅' if exists else '❌'
    if not exists:
        all_good = False
    print(f"  {status} {f} ({size:.1f} KB)")

print(f"\n{'All files ready! ✅' if all_good else 'Some files missing ❌'}")
print("\nDownload these files from Kaggle Output tab:")
for f in files:
    print(f"  → {f}")

Verifying saved files:
  ✅ label_encoder.pkl (0.3 KB)
  ✅ tokenizer.pkl (609.3 KB)
  ✅ baseline_bilstm.keras (48524.0 KB)
  ✅ bilstm_final.keras (46305.4 KB)
  ✅ bert_model.pt (427760.7 KB)

All files ready! ✅

Download these files from Kaggle Output tab:
  → label_encoder.pkl
  → tokenizer.pkl
  → baseline_bilstm.keras
  → bilstm_final.keras
  → bert_model.pt


In [17]:
# Cell 9 - Generate emotion_model.py
# This is the file your teammates import into app.py

code = '''
import numpy as np
import pickle
import torch
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import BertTokenizer, BertForSequenceClassification

# ── Constants ──────────────────────────────────────────────
MAX_SEQ_LEN  = 80
VOCAB_SIZE   = 30000
EMOTIONS     = ["Bored", "Confident", "Confused", "Curious", "Frustrated"]
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load artifacts once at import time ─────────────────────
with open("models/tokenizer.pkl", "rb") as f:
    _tokenizer = pickle.load(f)

with open("models/label_encoder.pkl", "rb") as f:
    _le = pickle.load(f)

# BiLSTM
_bilstm = load_model("models/bilstm_final.keras")

# BERT
_bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
_bert = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5,
    ignore_mismatched_sizes=True
)
_bert.load_state_dict(
    torch.load("models/bert_model.pt", map_location=DEVICE)
)
_bert.to(DEVICE)
_bert.eval()

print("emotion_model.py loaded successfully ✅")

# ── Core Function ───────────────────────────────────────────
def predict_emotion(text: str, model: str = "bert") -> dict:
    """
    Predict emotion from text.
    
    Args:
        text  : student input string
        model : "bert" or "bilstm"
    
    Returns:
        {
            "emotion"       : "Confused",
            "confidence"    : 0.87,
            "all_scores"    : {"Bored": 0.03, ...},
            "mixed_emotions": ["Confused", "Curious"]
        }
    """
    if model == "bert":
        probs = _predict_bert(text)
    else:
        probs = _predict_bilstm(text)

    top_idx        = int(np.argmax(probs))
    emotion        = _le.classes_[top_idx]
    confidence     = float(probs[top_idx])
    all_scores     = {_le.classes_[i]: round(float(probs[i]), 4)
                      for i in range(len(_le.classes_))}
    mixed_emotions = _get_mixed(probs)

    return {
        "emotion"       : emotion,
        "confidence"    : round(confidence, 4),
        "all_scores"    : all_scores,
        "mixed_emotions": mixed_emotions
    }


def _predict_bert(text: str) -> np.ndarray:
    enc = _bert_tokenizer(
        text,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = _bert(input_ids=input_ids,
                       attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return probs


def _predict_bilstm(text: str) -> np.ndarray:
    seq = _tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_SEQ_LEN,
                        padding="post", truncating="post")
    probs = _bilstm.predict(pad, verbose=0)[0]
    return probs


def _get_mixed(probs: np.ndarray, threshold: float = 0.20) -> list:
    """Return emotions with probability above threshold."""
    mixed = [_le.classes_[i] for i, p in enumerate(probs)
             if p >= threshold]
    return mixed if len(mixed) > 1 else [_le.classes_[int(np.argmax(probs))]]


# ── Quick Test ──────────────────────────────────────────────
if __name__ == "__main__":
    tests = [
        "I have no idea what recursion means no matter how many times I read it",
        "I finally understand how neural networks work and it all makes sense now",
        "I have been stuck on this bug for five hours and want to give up",
        "I wonder how transformers handle very long documents",
        "This lecture is just repeating last week with nothing new"
    ]
    expected = ["Confused", "Confident", "Frustrated", "Curious", "Bored"]

    print("\\nTesting predict_emotion():")
    print("─" * 60)
    for text, exp in zip(tests, expected):
        result = predict_emotion(text, model="bert")
        match  = "✅" if result["emotion"] == exp else "❌"
        print(f"{match} Expected: {exp:12} Got: {result['emotion']:12} "
              f"Conf: {result['confidence']:.2f}")
        print(f"   Mixed: {result['mixed_emotions']}")
        print(f"   Text:  {text[:55]}...")
        print()
'''

# Write to file
with open('emotion_model.py', 'w') as f:
    f.write(code)

print("emotion_model.py generated! ✅")
print("\nThis file is your handoff to teammates.")
print("They import it like this:")
print()
print("  from emotion_model import predict_emotion")
print()
print("  result = predict_emotion('I am confused about recursion')")
print("  # Returns:")
print("  # {")
print("  #   'emotion': 'Confused',")
print("  #   'confidence': 0.87,")
print("  #   'all_scores': {'Bored': 0.02, ...},")
print("  #   'mixed_emotions': ['Confused', 'Curious']")
print("  # }")

emotion_model.py generated! ✅

This file is your handoff to teammates.
They import it like this:

  from emotion_model import predict_emotion

  result = predict_emotion('I am confused about recursion')
  # Returns:
  # {
  #   'emotion': 'Confused',
  #   'confidence': 0.87,
  #   'all_scores': {'Bored': 0.02, ...},
  #   'mixed_emotions': ['Confused', 'Curious']
  # }


In [18]:
# Cell 10 - Zip everything for easy download
import zipfile
import os

files = [
    'label_encoder.pkl',
    'tokenizer.pkl',
    'bilstm_final.keras',
    'baseline_bilstm.keras',
    'bert_model.pt',
    'emotion_model.py'
]

with zipfile.ZipFile('emotion_models.zip', 'w') as zipf:
    for f in files:
        if os.path.exists(f):
            zipf.write(f)
            print(f"  Added: {f}")
        else:
            print(f"  Missing: {f} ❌")

size = os.path.getsize('emotion_models.zip') / (1024*1024)
print(f"\nZip created: emotion_models.zip ({size:.1f} MB) ✅")
print("Now download emotion_models.zip from Output tab")

  Added: label_encoder.pkl
  Added: tokenizer.pkl
  Added: bilstm_final.keras
  Added: baseline_bilstm.keras
  Added: bert_model.pt
  Added: emotion_model.py

Zip created: emotion_models.zip (510.9 MB) ✅
Now download emotion_models.zip from Output tab


In [1]:
# Cell 10 - Save custom dataset as Excel
import pandas as pd

custom_dataset = {
    "Confused": [
        "I've read this chapter three times and still don't understand what recursion actually means.",
        "I have no idea what the professor was trying to explain in today's lecture.",
        "The formula makes no sense to me no matter how many times I look at it.",
        "I thought I understood derivatives but now I'm completely lost again.",
        "Can someone explain what a pointer is because I genuinely don't get it.",
        "I don't know where to even begin with this assignment.",
        "The diagram in the textbook is making things more confusing not less.",
        "I understood the first part but completely lost track after that.",
        "What is the difference between a stack and a queue I keep mixing them up.",
        "I followed the example step by step but my answer is still wrong.",
        "The lecture notes don't match what the textbook says and I don't know which to follow.",
        "I keep getting a different answer every time I solve this problem.",
        "The explanation in class made sense but now at home I can't reproduce it.",
        "I have no clue what this error message means or how to fix it.",
        "I mixed up the concepts again and now my entire solution is wrong.",
        "The more I study this the less I feel I understand it.",
        "I thought I had the right approach but the output doesn't match expected result.",
        "I read the definition five times and I still can't put it in my own words.",
        "I don't understand why my code compiles but gives the wrong output.",
        "The slides say one thing and the tutorial says another thing entirely.",
        "I followed the formula exactly but I have no idea where the constant comes from.",
        "I'm not sure whether to use a for loop or a while loop in this case.",
        "The teacher skipped a step and now I can't fill in the gap myself.",
        "I genuinely don't understand what this question is even asking me to do.",
        "I know the steps individually but I don't understand how they connect.",
        "I have been staring at this proof for an hour and it still doesn't click.",
        "I don't know what base case to use for this recursive function.",
        "My professor uses different notation than the textbook and it's throwing me off.",
        "I understand the theory but have no idea how to apply it to this problem.",
        "I'm not sure if I'm supposed to normalize the data before or after splitting.",
        "The pseudocode makes sense but I can't translate it to actual code.",
        "I can't figure out the time complexity of this nested loop structure.",
        "I cannot figure out the difference between supervised and unsupervised here.",
        "I'm confused about whether this counts as overfitting or underfitting.",
        "I don't understand what the loss function is actually measuring in this model.",
        "I keep confusing precision and recall no matter how many times I study them.",
        "I cannot understand why we use softmax at the output layer specifically.",
        "I thought Boolean logic was simple but this truth table is destroying me.",
        "I can't figure out what this lambda function is doing in this code.",
        "The concept of inheritance makes no sense to me in object oriented programming.",
        "I don't understand why we pad sequences to the same length.",
        "I cannot figure out when to use mean squared error versus cross entropy loss.",
        "I don't know what it means for a matrix to be singular.",
        "I'm confused about why we need activation functions in neural networks.",
        "I don't understand how backpropagation actually updates the weights.",
        "I can't figure out what dropout actually does during training.",
        "I don't understand why my accuracy is high but my F1 score is low.",
        "I'm confused about why we use log probabilities instead of regular probabilities.",
        "I'm completely lost on what attention mechanism does in transformers.",
        "I can't figure out why my model performs well on training but poorly on new data.",
    ],
    "Curious": [
        "I wonder why neural networks need so many layers to solve simple problems.",
        "What would happen if we applied this algorithm to a completely different field.",
        "I'm interested in understanding how search engines rank pages so accurately.",
        "Can this same technique be used to solve problems in biology or medicine.",
        "I want to know what happens inside the model when it makes a wrong prediction.",
        "I'm fascinated by how computers can generate realistic images from text.",
        "What is the math behind how GPS calculates your exact position so quickly.",
        "I wonder if there's a faster way to solve this that nobody has discovered yet.",
        "How do recommendation systems know what I want to watch before I search.",
        "I'd love to understand the history of how this algorithm was first invented.",
        "What would this graph look like if we extended it to three dimensions.",
        "I'm curious whether this approach would still work with much larger datasets.",
        "Why do some machine learning models perform better on certain types of data.",
        "I want to explore what happens when you change each variable in this equation.",
        "How does autocomplete predict the next word so accurately in modern keyboards.",
        "I'm wondering if the same principle applies to quantum computing as well.",
        "What are the real world applications of this seemingly abstract concept.",
        "Can we combine two different algorithms to get better results than either alone.",
        "I'm curious about how self driving cars make split second decisions in traffic.",
        "I wonder how scientists first proved that this mathematical relationship was true.",
        "Is there a way to visualize what hidden layers of a neural network are learning.",
        "I want to explore whether emotions can really be detected accurately from text.",
        "How do language models handle sarcasm and irony when classifying sentiment.",
        "What happens to model performance when you remove one feature at a time.",
        "Can this classification problem be reframed as a regression problem instead.",
        "I'd like to experiment with different activation functions and compare results.",
        "I'm curious whether humans and AI make mistakes on the same types of examples.",
        "I want to understand how attention heads in BERT specialize for different tasks.",
        "Is there a mathematical proof that gradient descent always finds the minimum.",
        "I'm wondering how the brain's learning mechanism compares to backpropagation.",
        "What is the most efficient sorting algorithm ever discovered and why.",
        "Can reinforcement learning be applied to solve real world supply chain problems.",
        "I want to explore how music can be generated using deep learning models.",
        "I'm interested in how natural language processing handles multiple languages.",
        "How do scientists measure the intelligence or capability of an AI system.",
        "Can graph neural networks be used to model social relationships.",
        "I'm curious about how the human brain solves problems that stump AI systems.",
        "What would it take to build an AI that truly understands context like humans.",
        "How do AI systems learn to play games without being taught the rules explicitly.",
        "What are the ethical implications of using AI to make decisions about people.",
        "Can the same neural network architecture work for both images and text.",
        "What is the role of randomness in machine learning and why does it matter.",
        "Can we use machine learning to predict which students will struggle early.",
        "I wonder what role emotion detection could play in online learning platforms.",
        "How would a model trained on one language perform when tested on another.",
        "I'm curious about what limitations current AI models have that researchers work on.",
        "What is the relationship between information theory and machine learning.",
        "Can we use unsupervised learning to discover hidden patterns in student behavior.",
        "I wonder how different cultures encode emotion differently in their language.",
        "How do AI systems generalize from a few examples the way children naturally do.",
    ],
    "Frustrated": [
        "I have been trying to fix this bug for five hours and I still can't find it.",
        "No matter what I do my code keeps throwing the same error over and over.",
        "I studied for this exam for two weeks and still failed it completely.",
        "The professor explains things so fast that I can never keep up in class.",
        "I asked for help three days ago and still haven't received any response.",
        "I keep making the same mistake even after correcting it multiple times.",
        "This assignment is due in two hours and I haven't even started because I'm stuck.",
        "I feel like I'm the only one in the class who doesn't understand this.",
        "My code was working yesterday and now it's broken for no apparent reason.",
        "I've watched five tutorial videos and none of them explain it clearly enough.",
        "I spent the whole weekend on this project and it still doesn't work.",
        "Every time I fix one bug another one appears somewhere else.",
        "I gave the exact same answer as my classmate and they got full marks I got zero.",
        "I've been stuck on this one problem for so long I can't think straight anymore.",
        "The grading rubric is so vague that I have no idea what the teacher wants.",
        "I followed all the instructions exactly and my output is still completely wrong.",
        "I've asked the same question in class twice and the teacher just keeps moving on.",
        "My laptop crashed right before I could save my completed assignment.",
        "I keep running out of time on tests even though I know the material.",
        "The example in class was easy but the homework questions are impossibly harder.",
        "I practiced this problem type twenty times and still got it wrong on the test.",
        "My group members are not contributing and I'm doing all the work alone.",
        "The feedback on my last assignment was unhelpful and I don't know how to improve.",
        "I feel like I'm going in circles and making no progress at all.",
        "My development environment broke and I've spent more time setting it up than coding.",
        "The online portal keeps crashing when I try to submit my assignment.",
        "I feel like I'm falling further and further behind no matter how hard I try.",
        "My test anxiety is so bad that I forget everything the moment the exam starts.",
        "I've spent more time debugging than actually writing code today.",
        "The deadline was moved up without any warning and now I can't finish in time.",
        "I keep second guessing my answers and changing them from right to wrong.",
        "The sample solution uses a method we were never taught and it makes no sense.",
        "I'm putting in more hours than anyone else and still getting the lowest grades.",
        "I can't focus because I'm so stressed about how far behind I am.",
        "I've read this paragraph twenty times and it still doesn't make any sense.",
        "My model keeps overfitting no matter what regularization technique I try.",
        "I've been trying to install this library for two hours and keep hitting errors.",
        "I finally got my code working and then accidentally deleted the whole file.",
        "I've tried every approach I can think of and none of them work.",
        "I stayed up all night and the submission portal rejected my file format.",
        "I keep losing marks for things that were never mentioned in the requirements.",
        "I feel completely demoralized after working this hard and still failing.",
        "My partner dropped the course and now I have to redo their half of the project.",
        "The model training keeps crashing halfway through with a memory error.",
        "I've been working on this so long I can't even tell if it makes sense anymore.",
        "Every time I think I've solved the problem I discover a new edge case.",
        "I've tried every solution on Stack Overflow and none fix my specific problem.",
        "I feel invisible in this class because my questions are always ignored.",
        "I spent six hours on a problem that turned out to be a one character typo.",
        "I feel like I have to figure everything out alone with no support from anyone.",
    ],
    "Confident": [
        "I finally understand how recursion works and it actually makes total sense now.",
        "I solved that problem on my own without looking at any hints and I'm proud.",
        "I explained this concept to my study group and they all said it was very clear.",
        "I finished the assignment two days early and I'm happy with how it turned out.",
        "I got full marks on the test and I genuinely feel like I earned every point.",
        "I can now solve this type of problem in under two minutes consistently.",
        "I understand this topic well enough to teach it to someone else.",
        "I reviewed my old notes and I'm amazed by how much I've learned this semester.",
        "I attempted the hardest problem on the problem set and actually got it right.",
        "I feel completely prepared for tomorrow's exam and I'm not anxious at all.",
        "I made a mistake last week but I fully understand now why I was wrong.",
        "I can look at any problem of this type and immediately know the right approach.",
        "I built the entire project from scratch and it works exactly as intended.",
        "I understand not just what to do but also why each step is necessary.",
        "I feel like this topic has finally clicked after weeks of struggling with it.",
        "I can debug my code systematically and efficiently now without panicking.",
        "I submitted my assignment and I'm confident there are no major errors.",
        "I understand the underlying theory behind this algorithm not just how to use it.",
        "I walked into this exam feeling calm and walked out knowing I did well.",
        "I can now implement this from memory without needing to look anything up.",
        "I understand the trade-offs between different approaches to this problem.",
        "I reviewed every topic on the syllabus and feel genuinely ready.",
        "I can spot errors in other people's code quickly because I understand it so well.",
        "I feel confident that my solution is both correct and efficient.",
        "I just finished the most challenging assignment of the semester and nailed it.",
        "I can apply this concept to problems I've never seen before without guidance.",
        "I understand how all the topics in this course connect to each other now.",
        "I presented my project to the class and answered every question with confidence.",
        "I know exactly what went wrong in my last test and how to avoid it next time.",
        "I feel a real sense of mastery over this material after all the hard work.",
        "I can explain this concept simply and clearly to anyone who asks.",
        "I built a working model from scratch and it performs better than the baseline.",
        "I feel genuinely prepared to use these skills in a real world job.",
        "I understand this well enough to adapt it when the requirements change.",
        "I can write clean and readable code that other people can easily understand.",
        "I reviewed my solution multiple times and I'm satisfied it's completely correct.",
        "I feel strong in this subject now compared to how lost I was at the beginning.",
        "I answered the bonus question correctly and that means a lot to me.",
        "I can now solve problems under time pressure without making careless errors.",
        "I feel like I've developed real intuition for this type of problem.",
        "I understand the assumptions behind this model and when they might break down.",
        "I built a complete pipeline from data preprocessing to model evaluation.",
        "I can identify which algorithm is best suited for a given problem type.",
        "I feel confident that my code handles edge cases correctly.",
        "I understand why this approach works mathematically not just empirically.",
        "I got better results than expected and I know exactly why the model performed well.",
        "I feel like I've genuinely mastered this topic and I'm excited to use it.",
        "I can reproduce results from research papers and understand every step.",
        "I know how to tune hyperparameters effectively based on what I observe.",
        "I feel ready to tackle more advanced topics because my foundation is solid.",
    ],
    "Bored": [
        "This lecture is just repeating everything from last week with no new content.",
        "I already know everything being covered in this class and it's hard to stay engaged.",
        "The professor is reading directly from the slides without adding any insight.",
        "I finished this assignment in ten minutes but we were given two weeks to do it.",
        "This topic is so straightforward I can't understand why we're spending so long on it.",
        "I've been sitting here for an hour and nothing new has been introduced.",
        "I mastered this content in high school and now we're covering it again.",
        "The pace of this class is so slow that I find myself drifting off constantly.",
        "I keep checking the clock because this lecture feels like it's going nowhere.",
        "This problem set is just repetition with slightly different numbers every time.",
        "I could have completed this entire module in one afternoon instead of three weeks.",
        "The content stopped being challenging for me about two weeks ago.",
        "I know the answer before the teacher even finishes asking the question.",
        "This class moves so slowly that I spend most of it reading ahead in the textbook.",
        "I've already read the entire textbook and these lectures don't go beyond it.",
        "I find myself solving unrelated problems during lectures just to stay mentally active.",
        "This assignment feels like busywork rather than something that develops real skills.",
        "I'm not learning anything new in this course and I'm starting to lose interest.",
        "The examples in today's class were all identical to the ones from last week.",
        "I finished my exam early and spent thirty minutes just waiting to hand it in.",
        "The level of this course is much lower than I expected coming in.",
        "I feel like I'm just going through the motions without gaining any new understanding.",
        "I've heard this explanation so many times I can recite it word for word.",
        "I don't feel challenged by any of the tasks we're being assigned.",
        "I sat through the whole lecture and didn't write down a single new thing.",
        "I feel completely unchallenged and it's making it really hard to stay motivated.",
        "I could write the answer to this question in my sleep at this point.",
        "I'm attending every class but I don't feel like any of them are adding value.",
        "I keep hoping the next topic will be harder but it never is.",
        "This quiz was so easy that I'm not sure what the point of it was.",
        "I've been in this study session for two hours and reviewed content I already know.",
        "The assignments in this course test memory not actual thinking or problem solving.",
        "I feel like I'm doing the same thing over and over again with nothing changing.",
        "I find myself correcting the professor's examples internally without saying anything.",
        "The course description promised advanced content but this feels very entry level.",
        "I completed the weekly readings in one sitting because there was nothing new.",
        "I feel like the course is designed for people who have never seen this before.",
        "I don't need to take notes anymore because nothing being said is new to me.",
        "I answered every practice question correctly without even trying.",
        "I feel like I'm waiting for the course to catch up to my current knowledge level.",
        "I've already implemented this type of project before so this one holds no novelty.",
        "The lecture is covering basics I learned independently months ago.",
        "I feel like my time would be better spent learning something more advanced.",
        "The tutorial exercises are so simple they don't require any actual thought.",
        "I sat through a ninety minute class and absorbed exactly zero new information.",
        "I finished every item on the assignment checklist in under an hour.",
        "I have no motivation to do the readings because I already know what they'll say.",
        "I'm learning more from my own side projects than from this class.",
        "The problem sets feel like they were designed for a much more introductory level.",
        "I feel like I'm not being pushed to grow at all in this course.",
    ]
}

# Convert to DataFrame
rows = []
for emotion, sentences in custom_dataset.items():
    for sentence in sentences:
        rows.append({
            'text': sentence,
            'emotion': emotion,
            'source': 'custom_academic'
        })

df = pd.DataFrame(rows)

# Add label column
from sklearn.preprocessing import LabelEncoder
le_temp = LabelEncoder()
df['label'] = le_temp.fit_transform(df['emotion'])

print("Dataset summary:")
print(df['emotion'].value_counts())
print(f"\nTotal sentences: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Save as Excel
df.to_excel('emotion_text_dataset.xlsx', index=False)
print("\nSaved as emotion_text_dataset.xlsx ✅")

# Also save as CSV backup
df.to_csv('emotion_text_dataset.csv', index=False)
print("Saved as emotion_text_dataset.csv ✅")

# Preview
print("\nSample rows:")
print(df.groupby('emotion').first()[['text','label']])

Dataset summary:
emotion
Confused      50
Curious       50
Frustrated    50
Confident     50
Bored         50
Name: count, dtype: int64

Total sentences: 250
Columns: ['text', 'emotion', 'source', 'label']

Saved as emotion_text_dataset.xlsx ✅
Saved as emotion_text_dataset.csv ✅

Sample rows:
                                                         text  label
emotion                                                             
Bored       This lecture is just repeating everything from...      0
Confident   I finally understand how recursion works and i...      1
Confused    I've read this chapter three times and still d...      2
Curious     I wonder why neural networks need so many laye...      3
Frustrated  I have been trying to fix this bug for five ho...      4
